<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons Licence" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">Logistic Regression on MNIST</span> by <span xmlns:cc="http://creativecommons.org/ns#" property="cc:attributionName">Marc de Kamps and University of Leeds</span> is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.

# Logistic regression on MNIST, by hand

**COMP3620 Neural Networks and Deep Learning — Understanding Deep Learning, Chapters 5 and 6**

This notebook trains a **multiclass logistic-regression (softmax) classifier** on **MNIST**, the dataset of handwritten digits. There is no deep-learning framework in the model or the training: the softmax, the gradient and the gradient-descent update are written out in NumPy, one line each. PyTorch is used only to download the data.

It runs on any laptop; no GPU needed. A full run takes a few minutes, most of it in the training loop.

**What it demonstrates**
1. How to get MNIST through `torchvision` and turn it into a **design matrix** with one image per row.
2. How to split the data into a **training set (80%)** and a **test set (20%)** with plain Python slicing.
3. A literal implementation of **stochastic gradient descent** for the multiclass cross-entropy loss.
4. The difference between an untrained classifier (chance level) and a trained one, measured on the test set.

## 1. Data: MNIST from torchvision

MNIST is a large collection of handwritten numerals. The images are grey-scale, $28 \times 28$ pixels, and each image comes with a label, which can take any of the ten values '0', '1', …, '9': a human judgement about which numeral the image is supposed to represent.

We use `torchvision.datasets.MNIST`. The download is about 12&nbsp;MB; it is cached in `./data` after the first run. `ToTensor()` turns each image into a PyTorch tensor with pixel values in $[0, 1]$. For practical purposes you can treat tensors as NumPy arrays for now.

In [ ]:
# Source: COMP3611-2025/Labs/COMP3611 - Logistic Regression on MNIST.ipynb
# (Marc de Kamps, University of Leeds, CC BY 4.0), adapted for COMP3620.
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

np.random.seed(0)

transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
print(f'MNIST: {len(dataset)} images')

## 2. A first look at the data

It is easy to inspect the dataset. Each element is a pair:

In [ ]:
print(dataset[0])

Careful inspection of the output shows a pair of two elements. Let us first consider the second element:

In [ ]:
print(dataset[0][1])

This is a single numeral: possibly the label of the image, which would then be the first element. Let's explore this hypothesis.

In [ ]:
# In general you have to be careful in converting tensors to NumPy arrays.
# A tensor may live on the GPU, and it can carry information that isn't data,
# such as gradients. First move it to the CPU, detach the data, then convert.
np_arr = dataset[0][0].to('cpu').detach().numpy()
print(np_arr.shape)

It is tempting to run `imshow` directly on `np_arr`. It won't work, and the shape should give you a clue as to why not.

In [ ]:
print(np_arr[0].shape)
nr_pix = np_arr[0].shape[0]

In [ ]:
plt.imshow(np_arr[0])
plt.show()

That's better! Do the image and label match? Let's try another pattern.

In [ ]:
print(dataset[122][1])
plt.imshow(dataset[122][0][0])
plt.show()

## 3. The design matrix and an 80/20 split

Flatten every image into a row of $28 \cdot 28 = 784$ pixels. Stacked, the rows form the **design matrix**.

In [ ]:
design = np.array([x.to('cpu').detach().numpy().flatten() for x, _ in dataset])
labels = np.array([y for _, y in dataset])
print(design.shape, labels.shape)

Are the rows indeed images?

In [ ]:
img22 = np.array(design[22])   # copy it
img22.shape = nr_pix, nr_pix
plt.imshow(img22)
plt.title(f'row 22, label {labels[22]}')
plt.show()

Looks that way, so `design` is a design matrix. Convert the labels into **one-hot target vectors**: row $i$ of `t` is all zeros except for a 1 in the column of label $i$.

In [ ]:
nr_classes = 10
t = np.zeros((design.shape[0], nr_classes))
for i, ind in enumerate(labels):
    t[i][ind] = 1.
print(t.shape, '| first row:', t[0], '| label:', labels[0])

Add a constant 1 to each pattern, so that the bias becomes one more weight:

In [ ]:
ones = np.ones((design.shape[0], 1))
designbias = np.concatenate((design, ones), axis=1)
print(designbias.shape)

Now split the data: the **first 80%** of the rows for training, the **last 20%** for testing. This is plain Python slicing, `[:n_train]` and `[n_train:]`, applied in the same way to the patterns, the targets and the labels, so that row $i$ still matches across all three. The test set is never used for training; it only measures how well the classifier does on images it has not seen.

Slicing like this is only fair if the rows are not ordered by class. The class counts printed below show that both parts contain all ten digits in similar proportions.

In [ ]:
n_train = int(0.8 * designbias.shape[0])

designbias_train, designbias_test = designbias[:n_train], designbias[n_train:]
t_train,          t_test          = t[:n_train],          t[n_train:]
labels_train,     labels_test     = labels[:n_train],     labels[n_train:]

print(f'training set: {designbias_train.shape[0]} patterns | test set: {designbias_test.shape[0]} patterns')
print('class counts, training:', np.bincount(labels_train))
print('class counts, test    :', np.bincount(labels_test))

## 4. The model: softmax regression

Create the weights. Each image (784 pixels plus the constant 1) connects to each of 10 output nodes, one per class, so the weights form a $785 \times 10$ matrix.

In [ ]:
weights = np.random.normal(0., 1., size=(nr_pix * nr_pix + 1, nr_classes))
print(weights.shape)

A naive implementation of softmax can run into a numerical problem: $e^z$ overflows for large $z$. Have a look at the Stack Overflow discussion linked in the code below.

The gradient of the multiclass cross-entropy loss $L$ with respect to the weights $\mathbf{w}_j$ of output node $j$ is
$$
\frac{\partial L}{\partial \mathbf{w}_j} = \sum^I_{i=1}(y_{ij} - t_{ij}) \boldsymbol{\Phi}_i ,
$$
where $\boldsymbol{\Phi}_i$ is pattern $i$ (a row of `designbias`), $y_{ij}$ the softmax output of node $j$ for that pattern, $t_{ij}$ its target, and $I$ the number of training patterns. With the help of `np.outer`, the gradient for a single pattern can be expressed very tersely in Python.

In [ ]:
# A suitable softmax implementation was taken from this discussion:
# https://stackoverflow.com/questions/42599498/numercially-stable-softmax
# (sic)

def stableSoftmax(x):
    z = x - np.max(x, axis=-1, keepdims=True)
    numerator = np.exp(z)
    denominator = np.sum(numerator, axis=-1, keepdims=True)
    softmax = numerator / denominator
    return softmax

def update(designbias, weights):
    return stableSoftmax(designbias.dot(weights))

# The gradient is calculated over a single input pattern. This is a 785 x 10 matrix.
# Doing this over the entire dataset would require 48000 of them, which may lead to
# memory problems.

def gradient(target, weights, designbias, i):
    ms = np.outer(designbias[i], stableSoftmax(designbias[i].dot(weights)) - target[i])
    return ms



## 5. Before training: chance level

We use a very crude measure: we count how many test patterns are classified in agreement with their label, taking the class with the largest softmax output as the prediction.

In [ ]:
def correctlyClassified(labels, designbias, weights):
    outcomes = update(designbias, weights)
    outcomelabels = np.array([np.argmax(x) for x in outcomes])
    difftest = outcomelabels - labels
    return np.count_nonzero(difftest == 0), outcomes.shape[0]

correctlyClassified(labels_test, designbias_test, weights)

With random weights the result is around chance level, as it should be. Chance is 10% for ten classes, but you may well see less: random weights tend to favour one or two classes for almost every image.

## 6. Training by stochastic gradient descent

Each step picks one random pattern **from the training set**, computes the gradient for that pattern, and moves the weights a small step (learning rate $r$) against it.

In [ ]:
nrdatapoints = designbias_train.shape[0]
r = 0.02
for i in range(50000):
    if i % 1000 == 0: print(i)
    weights -= r * gradient(t_train, weights, designbias_train, np.random.randint(0, nrdatapoints - 1))

## 7. Evaluation on the test set

The classifier has now been trained on the training set only. The test set gives an honest estimate of how it does on new images.

In [ ]:
n_correct, n_total = correctlyClassified(labels_test, designbias_test, weights)
print(f'test set: {n_correct} of {n_total} patterns correctly classified '
      f'(accuracy {n_correct / n_total:.3f})')

In [ ]:
# Three wrongly classified test patterns, with the true and the predicted label
predicted = np.argmax(update(designbias_test, weights), axis=1)
wrong = np.flatnonzero(predicted != labels_test)
print(f'{len(wrong)} of {len(labels_test)} test patterns are wrongly classified')

picks = np.random.choice(wrong, size=3, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
for ax, i in zip(axes, picks):
    ax.imshow(designbias_test[i, :-1].reshape(nr_pix, nr_pix))   # drop the bias column
    ax.set_title(f'test pattern {i}\nlabel {labels_test[i]}, predicted {predicted[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()